# Week 1 · Tutorial 2 — Dates and times, input vs. target, tidy data

**Covers videos 1.4 – 1.6:** dates and times · input and target variables · tidy data.

**What you will do**
1. Parse **ambiguous date strings** from an EV-charging export and deal with the autumn **daylight-saving** change.
2. Ask of every column: *when did this value become known?* (substation load, a leakage hunt).
3. Reshape an **untidy wide table** of PV inverters and **join** it without silently changing the row count.

In every **✏️ Your turn** block you make a small, clearly described change and run the cell. The results are what the quiz asks about.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Dates and times

`ev_sessions.csv` is an export from a charging network in Norway. **One row is one charging session.** The start time
is written in *local* time as `dd.mm.yyyy HH:MM`, and the column `tz` says whether summer time (`CEST`, UTC+2) or winter
time (`CET`, UTC+1) applied. On **Sunday 29 October 2023** the clocks went back from 03:00 to 02:00, so the local
hour 02:00–02:59 happened **twice**.

In [ ]:
ev = pd.read_csv("data/ev_sessions.csv")
print(ev.shape)
ev.head()

A text column is not a time stamp. `03.11.2023` is 3 November in Norway, but a month-first format would read it as *11 March*,
and `26.10.2023` would not be a valid date at all.

### ✏️ Your turn 1

Parse `start_local` properly.

The starter reads the text with the **month-first** format `%m.%d.%Y %H:%M` and silently turns failures into `NaT`.
Run it once as it is and note how many sessions fail to parse. Then change the format to **day first**
(`"%d.%m.%Y %H:%M"`) and run again.

The cell also counts the sessions that started on **29 October 2023** (local date).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
ev["start"] = pd.to_datetime(ev["start_local"], format="%m.%d.%Y %H:%M", errors="coerce")
```
by
```python
ev["start"] = pd.to_datetime(ev["start_local"], format="%d.%m.%Y %H:%M", errors="coerce")
```

In [ ]:
ev["start"] = pd.to_datetime(ev["start_local"], format="%m.%d.%Y %H:%M", errors="coerce")   # TODO: day-first format

print("sessions that failed to parse:", ev["start"].isna().sum())
print("months present:", sorted(ev["start"].dt.month.unique()))
print("sessions on 29 Oct 2023 (local date):", (ev["start"].dt.date == pd.Timestamp("2023-10-29").date()).sum())
print("first session:", ev["start"].min(), " last session:", ev["start"].max())

### Store UTC, display local

Local times with daylight saving are not a continuous axis. We convert to **UTC**. To localise a time stamp pandas must
know which of the two possible 02:xx hours is meant. The `tz` column tells us (`CEST` = first pass, daylight saving *on*).

### ✏️ Your turn 2

Resolve the ambiguity with the information in the `tz` column.

1. In the `TODO` line pass `ambiguous=(ev["tz"] == "CEST").values` to `tz_localize` (`True` = daylight saving).
   Run the cell first *without* the change and note how many sessions are lost (`NaT`); then make the change.
2. The cell converts to UTC and counts sessions and delivered energy per **UTC** date.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
local = ev["start"].dt.tz_localize("Europe/Oslo", ambiguous="NaT")
```
by
```python
local = ev["start"].dt.tz_localize("Europe/Oslo", ambiguous=(ev["tz"] == "CEST").values)
```

In [ ]:
local = ev["start"].dt.tz_localize("Europe/Oslo", ambiguous="NaT")   # TODO: resolve the ambiguity using the tz column
ev["start_utc"] = local.dt.tz_convert("UTC")

print("rows with missing UTC time:", ev["start_utc"].isna().sum())
ev["utc_date"] = ev["start_utc"].dt.date.astype(str)
daily = ev.groupby("utc_date").agg(sessions=("session_id", "size"), energy_kwh=("energy_kwh", "sum"))
print(daily.round(1))

### ✏️ Your turn 3

How long is a local day on a daylight-saving change? Change the date in the `TODO` line.

The starter counts the hours of the local day **2023-10-29**. Change `day` to **2024-03-31** (the spring change, clocks
jump from 02:00 to 03:00) and run again; then try **2024-07-01** as a normal day.

In [ ]:
day = "2023-10-29"      # TODO: try "2024-03-31" and "2024-07-01"
hours = pd.date_range(day, periods=26, freq="h", tz="Europe/Oslo")
hours_in_day = (hours.normalize() == hours[0].normalize()).sum()
print(day, "has", hours_in_day, "hours")

## 2. Input and target — *when did this value become known?*

`substation_hourly.csv` has one row per hour for a distribution substation (November–December 2023).
**Question we want to answer:** *what will the load be in the next hour?*
The **target** is `load_next_hour_mw`. Every other column is a candidate **input** — but only if its value is known
**at prediction time**, i.e. at the end of the current hour.

In [ ]:
sub = pd.read_csv("data/substation_hourly.csv")
print(sub.shape)
sub.head()

| column | what it is |
|---|---|
| `load_mw` | load during the current hour |
| `temperature_c`, `hour`, `weekday` | current temperature and calendar information |
| `load_prev_hour_mw` | load one hour earlier |
| `daily_peak_mw` | highest hourly load **of the calendar day** |
| `peak_alarm_next_hour` | 1 if the next hour is among the 5 % highest-load hours |
| `temperature_forecast_c` | weather-service forecast for the next hour, issued at the start of the current hour |

### ✏️ Your turn 4

Mark which columns are legitimately available at prediction time.

In the dictionary below every column is marked `True`. Two of them should be `False` because they contain
information that is **not known** at the end of the current hour. Set those two to `False`, then run the cell: it
keeps only the allowed columns and prints their correlation with the target.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
    "daily_peak_mw": True,
    "peak_alarm_next_hour": True,
```
by
```python
    "daily_peak_mw": False,
    "peak_alarm_next_hour": False,
```

In [ ]:
known_at_prediction_time = {
    "load_mw": True,
    "temperature_c": True,
    "hour": True,
    "weekday": True,
    "load_prev_hour_mw": True,
    "daily_peak_mw": True,            # TODO: True or False?
    "peak_alarm_next_hour": True,     # TODO: True or False?
    "temperature_forecast_c": True,
}

inputs = [c for c, ok in known_at_prediction_time.items() if ok]
corr = sub[inputs + ["load_next_hour_mw"]].corr()["load_next_hour_mw"].drop("load_next_hour_mw")
print("number of inputs kept:", len(inputs))
print(corr.round(3).sort_values(ascending=False))
print("strongest input:", corr.abs().idxmax(), round(corr.abs().max(), 3))

### ✏️ Your turn 5

A *classification* version of the question: *"Will the next hour be a peak hour?"* Now the target is `peak_alarm_next_hour`.
Change the `TODO` so that the target is `"peak_alarm_next_hour"`.

The cell prints the share of peak hours (the *base rate*) and the average `load_mw` in peak vs. non-peak hours.

**What to change** — edit the cell exactly like this, then run it:

Change `target = "load_next_hour_mw"` into `target = "peak_alarm_next_hour"`

In [ ]:
target = "load_next_hour_mw"     # TODO: change the target to "peak_alarm_next_hour"

print("target:", target, "| type:", "classification" if sub[target].nunique() == 2 else "regression")
if sub[target].nunique() == 2:
    print("share of peak hours:", round(sub[target].mean(), 4))
    print(sub.groupby(target)["load_mw"].mean().round(2))

## 3. Tidy data — one variable per column, one observation per row

`pv_inverters_wide.csv` has **one row per inverter**, but twelve columns `jan … dec` that all hold the *same variable*
(energy produced in kWh). That is **untidy** — the month is hidden in the column name.

In [ ]:
wide = pd.read_csv("data/pv_inverters_wide.csv")
print(wide.shape)
wide.head(4)

### ✏️ Your turn 6

Make the table tidy: **one row per inverter per month**.

Change the `melt` call: give the month column the name `month` (`var_name="month"`) and the values the name `kwh`
(`value_name="kwh"`). Run it; you should have the columns `inverter_id`, `month`, `kwh`.

**What to change** — edit the cell exactly like this, then run it:

Change `long = wide.melt(id_vars="inverter_id")` into `long = wide.melt(id_vars="inverter_id", var_name="month", value_name="kwh")`

In [ ]:
long = wide.melt(id_vars="inverter_id")      # TODO: var_name="month", value_name="kwh"
print(long.shape)
print(list(long.columns))

Now attach the site and installed capacity from `inverter_info.csv` (one row should be one inverter).
**A left join should not change the row count.** Always check.

In [ ]:
info = pd.read_csv("data/inverter_info.csv")
print("rows in info:", len(info), "| distinct inverters in info:", info["inverter_id"].nunique())

### ✏️ Your turn 7

The info table has a **duplicated key**, so the join creates extra rows.

1. Run the cell once as it is and note the row counts before and after the join.
2. Then fix it: replace the `TODO` by `info = info.drop_duplicates("inverter_id")` *before* the merge, so the join
   is again one row per inverter-month. Run again.
3. The cell then computes the yearly specific yield (kWh per kWp) per inverter and averages it per site.

**What to change** — edit the cell exactly like this, then run it:

Change `# TODO: remove the duplicated key row(s) from `info` before joining` into `info = info.drop_duplicates("inverter_id")`

In [ ]:
# TODO: remove the duplicated key row(s) from `info` before joining

tidy = long.merge(info, on="inverter_id", how="left")
print("rows before join:", len(long), "| rows after join:", len(tidy))

per_inv = tidy.drop_duplicates(["inverter_id", "month"]).groupby(["site", "inverter_id"]).agg(
    kwh=("kwh", "sum"), kwp=("capacity_kwp", "first")).reset_index()
per_inv["kwh_per_kwp"] = per_inv["kwh"] / per_inv["kwp"]
by_site = per_inv.groupby("site")["kwh_per_kwp"].mean().sort_values(ascending=False)
print(by_site.round(0))
print("total kWh over all rows of the tidy table:", int(tidy["kwh"].sum()))

### ✏️ Your turn 8

With the tidy table, a monthly profile takes one line. Fill in the `TODO`: the **total kWh of all inverters** per
month (`tidy.groupby("month")["kwh"].sum()`). Then the cell prints the month with the highest total and the share of
the yearly total that falls in **jun, jul and aug** (3 decimals).

**What to change** — edit the cell exactly like this, then run it:

Change `monthly = pd.Series(dtype=float)` into `monthly = tidy.groupby("month")["kwh"].sum()`

In [ ]:
monthly = pd.Series(dtype=float)   # TODO

if len(monthly):
    print("best month:", monthly.idxmax())
    print("share jun+jul+aug:", round(monthly[["jun", "jul", "aug"]].sum() / monthly.sum(), 3))

## Summary

* Parse dates with an explicit format, store **UTC**, display local; a local day can have 23 or 25 hours.
* Ask of every column: **when did this value become known?** The target and its near-copies must never be inputs.
* Tidy tables make aggregations simple; after every join, **check the row count**.